# Notebook 05 — HarmBench Cross-Dataset Validation

## What this notebook does
Runs Llama-Guard-3-8B on HarmBench (post-dates ToxicChat, no training overlap)
and compares precision/recall to ToxicChat results. Confirms that the guard's
failure patterns are genuine generalisation gaps, not memorisation artefacts.

## Prerequisites
- GPU runtime (T4/A100)
- HuggingFace token for Llama-Guard-3-8B
- Nothing from Drive needed (HarmBench loaded directly from HuggingFace)

## Outputs saved to Drive
| File | Used by |
|---|---|
| `harmbench_validation.json` | reporting |

## Key finding
Precision=99.3% on HarmBench vs 49.1% on ToxicChat — failures are real gaps.

**Purpose:** Address training-data contamination concern.
If precision/recall on HarmBench is similar to ToxicChat (precision≈88%, recall≈51%),
it confirms the guard's failure patterns are genuine generalisation gaps, not
memorisation artefacts.

**CPU-only for analysis; GPU needed for guard inference.**

### Step 0 — get the repo onto this runtime

In [ ]:
import os, subprocess
from pathlib import Path

# ── EDIT THIS if you have a GitHub remote ────────────────────────────────────
GITHUB_URL = "https://github.com/yogijoshi86/SLMProject.git"
# ─────────────────────────────────────────────────────────────────────────────

TARGET = Path("/content/SLMProject")

if TARGET.is_dir() and (TARGET / "src" / "guardrail_audit").is_dir():
    print("Repo already present — pulling latest…")
    subprocess.run(["git", "-C", str(TARGET), "pull", "--ff-only"], check=True)

elif GITHUB_URL:
    print("Cloning from GitHub…")
    subprocess.run(["git", "clone", GITHUB_URL, str(TARGET)], check=True)
    print("Cloned to", TARGET)

else:
    # ── Google Drive fallback ─────────────────────────────────────────────────
    # Mount Drive once then point DRIVE_PATH at wherever you stored the folder.
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PATH = "/content/drive/MyDrive/SLMProject"   # adjust if needed
    if not Path(DRIVE_PATH).is_dir():
        raise FileNotFoundError(
            f"Could not find the repo at {DRIVE_PATH}. "
            "Either set GITHUB_URL above, or copy the SLMProject folder to your Drive "
            "and update DRIVE_PATH."
        )
    import shutil
    shutil.copytree(DRIVE_PATH, str(TARGET))
    print("Copied from Drive to", TARGET)

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/content/SLMProject")
assert (REPO_ROOT / "src" / "guardrail_audit").is_dir(), \
    f"src/guardrail_audit not found under {REPO_ROOT}. Did the previous cell succeed?"

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
# Pin exact versions proven compatible on Colab T4.
%pip install -q -e ".[quant,explainer,dev]" \
    "torch>=2.4.0" "torchvision>=0.19.0" \
    "transformers==4.44.2" \
    "accelerate==0.33.0" \
    "bitsandbytes>=0.45.0" \
    "numpy>=1.26,<2.0"

In [ ]:
# MUST RUN after INSTALL. Restarts the kernel so upgraded packages load fresh.
# After restart: skip this cell and the INSTALL cell, run from the next cell down.
import os, sys
# Sanity-check: if numpy is already broken, restart is definitely needed.
try:
    import numpy as np; np.random.seed(0)
    print("Packages loaded OK. Restarting to ensure clean state...")
except Exception as e:
    print(f"Detected stale package (numpy ABI mismatch or similar): {e}")
    print("Restarting now...")
os.kill(os.getpid(), 9)

### After restart — re-run from the LOCATE cell below

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/content/SLMProject")
assert (REPO_ROOT / "src" / "guardrail_audit").is_dir(), \
    f"src/guardrail_audit not found under {REPO_ROOT}. Did the previous cell succeed?"

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
# ── Connect to Google Drive ───────────────────────────────────────────────────
# Run once per Colab session. Mounts Drive and sets up shared artifact folder.
# All notebooks read/write artifacts to the same Drive path so state persists
# across sessions and is shared between notebooks without re-running upstream ones.
from google.colab import drive
from pathlib import Path
import os, shutil

try:
    drive.mount("/content/drive")
    MOUNTED = True
except Exception as e:
    print(f"Drive mount skipped ({e}) — artifacts will not persist across sessions.")
    MOUNTED = False

DRIVE_ARTIFACTS = "/content/drive/MyDrive/hf_cache/artifacts"
DRIVE_FIGURES   = "/content/drive/MyDrive/hf_cache/figures"
DRIVE_FORMS     = "/content/drive/MyDrive/hf_cache/study_forms"

if MOUNTED:
    Path(DRIVE_ARTIFACTS).mkdir(parents=True, exist_ok=True)
    Path(DRIVE_FIGURES).mkdir(parents=True, exist_ok=True)
    Path(DRIVE_FORMS).mkdir(parents=True, exist_ok=True)
    # Also redirect HuggingFace cache so 16GB model downloads persist
    HF_CACHE = "/content/drive/MyDrive/hf_cache"
    os.environ["HF_HOME"] = HF_CACHE
    os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
    print(f"Drive mounted ✓  artifacts={DRIVE_ARTIFACTS}")
else:
    DRIVE_ARTIFACTS = "artifacts"   # fallback: local only
    print("Drive not mounted — using local artifacts/ only")

In [ ]:
import numpy as np; np.random.seed(0)
print("packages OK")

In [ ]:
from guardrail_audit.utils import load_config, set_seed

# colab_smoke.yaml = 500 prompts + int8 (fits a free T4). Swap to default.yaml for full runs.
CONFIG = "config/colab_smoke.yaml"
cfg = load_config(CONFIG)
set_seed(cfg.seed)
cfg

### (Optional) Restore Drive cache to skip model download

In [ ]:
# ── Restore artifacts from Drive ─────────────────────────────────────────────
# Run this at the start of any notebook to reload outputs from prior notebooks
# without re-running them. Copies everything from Drive artifacts/ to local.
import shutil
from pathlib import Path

DRIVE_ARTIFACTS = "/content/drive/MyDrive/hf_cache/artifacts"
LOCAL_ARTIFACTS = Path("artifacts")
LOCAL_ARTIFACTS.mkdir(exist_ok=True)

if not Path(DRIVE_ARTIFACTS).exists():
    print("Drive artifacts folder not found — run notebook 01 first to create it.")
else:
    restored = []
    for f in Path(DRIVE_ARTIFACTS).glob("*"):
        dest = LOCAL_ARTIFACTS / f.name
        shutil.copy(f, dest)
        restored.append(dest.name)
    print(f"Restored {len(restored)} files from Drive:")
    for name in sorted(restored):
        print(f"  {name}")

In [ ]:
import os, torch

# Reduce CUDA memory fragmentation — set before any model load
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

assert torch.cuda.is_available(), (
    "No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4/A100 GPU, then rerun."
)
print("GPU:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
if free < 10e9:
    print("WARNING: < 10 GB free. Consider Runtime → Disconnect and delete runtime.")

### HuggingFace auth

In [ ]:
import os, getpass
from huggingface_hub import login
token = getpass.getpass("HuggingFace token: ")
os.environ["HF_TOKEN"] = token
login(token=token)

### Load HarmBench dataset

In [ ]:
from datasets import load_dataset

# HarmBench standard behaviours — 400 harmful prompts, diverse attack categories.
# walledai/HarmBench is a public mirror; original: Paul-Louis Pröve/harmbench
print("Loading HarmBench...")
try:
    ds_harm = load_dataset("walledai/HarmBench", split="train")
except Exception:
    ds_harm = load_dataset("McGill-NLP/HarmBench-100", split="test")

print(f"HarmBench rows: {len(ds_harm)}")
print("Columns:", ds_harm.column_names)
print("Sample:", ds_harm[0])

### Prepare sample — 500 prompts (300 harmful + 200 benign)

In [ ]:
import random
random.seed(42)

# Harmful prompts from HarmBench (ground truth = toxic)
harm_col = "behavior" if "behavior" in ds_harm.column_names else "prompt"
harmful_prompts = [
    {"text": r[harm_col], "gt_toxicity": 1}
    for r in ds_harm
    if r.get(harm_col) and len(r[harm_col].strip()) > 20
][:300]

# Benign prompts — use a clean instruction-following dataset as safe counterpart
try:
    ds_benign = load_dataset("tatsu-lab/alpaca", split="train")
    benign_col = "instruction"
except Exception:
    ds_benign = load_dataset("databricks/databricks-dolly-15k", split="train")
    benign_col = "instruction"

benign_prompts = [
    {"text": r[benign_col], "gt_toxicity": 0}
    for r in ds_benign
    if r.get(benign_col) and len(r[benign_col].strip()) > 20
    and not any(w in r[benign_col].lower() for w in ["kill", "harm", "illegal", "weapon"])
]
random.shuffle(benign_prompts)
benign_prompts = benign_prompts[:200]

all_prompts = harmful_prompts + benign_prompts
random.shuffle(all_prompts)

print(f"Total: {len(all_prompts)} prompts ({len(harmful_prompts)} harmful, {len(benign_prompts)} benign)")
print(f"Sample harmful: {harmful_prompts[0]['text'][:80]}")
print(f"Sample benign:  {benign_prompts[0]['text'][:80]}")

### Load Llama-Guard-3-8B

In [ ]:
from guardrail_audit.models import load_guard

guard = load_guard(cfg.model)
print("Loaded:", cfg.model.name, "| dtype:", cfg.model.dtype)

### Run guard on HarmBench sample

In [ ]:
from guardrail_audit.data import batched
from tqdm import tqdm

decisions = []
batch_size = cfg.extraction.batch_size

for chunk in tqdm(list(batched(all_prompts, batch_size)), desc="Running guard", unit="batch"):
    texts = [p["text"] for p in chunk]
    batch_decisions, _ = guard.classify_batch(texts)
    for prompt, decision in zip(chunk, batch_decisions):
        decisions.append({
            "text": prompt["text"],
            "gt_toxicity": prompt["gt_toxicity"],
            "guard_unsafe": decision.is_unsafe,
            "guard_categories": decision.categories,
        })

print(f"Processed {len(decisions)} prompts")
print(f"Guard flagged UNSAFE: {sum(1 for d in decisions if d['guard_unsafe'])}")

### Compute FP/FN metrics

In [ ]:
import json
from pathlib import Path

tp = sum(1 for d in decisions if d["guard_unsafe"] and d["gt_toxicity"] == 1)
fp = sum(1 for d in decisions if d["guard_unsafe"] and d["gt_toxicity"] == 0)
fn = sum(1 for d in decisions if not d["guard_unsafe"] and d["gt_toxicity"] == 1)
tn = sum(1 for d in decisions if not d["guard_unsafe"] and d["gt_toxicity"] == 0)

n = len(decisions)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy  = (tp + tn) / n
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("=== HarmBench Results ===")
print(f"N={n}  |  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"Accuracy:  {accuracy:.1%}")
print(f"Precision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"F1:        {f1:.3f}")

# ToxicChat reference (from experiments on full dataset)
print("\n=== ToxicChat Reference ===")
print("Accuracy:  77.6%")
print("Precision: 87.9%")
print("Recall:    51.0%")
print("F1:        0.646")

print("\n=== Interpretation ===")
delta_prec = abs(precision - 0.879)
delta_rec  = abs(recall - 0.510)
if delta_prec < 0.10 and delta_rec < 0.10:
    print("Similar performance across datasets (Δprecision<10%, Δrecall<10%).")
    print("Training contamination is unlikely to be driving ToxicChat results.")
else:
    print(f"Notable difference: Δprecision={delta_prec:.1%}, Δrecall={delta_rec:.1%}")
    print("Report both datasets and discuss the discrepancy in the paper.")

### Visualise comparison

In [ ]:
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "F1"]
toxic_vals = [0.776, 0.879, 0.510, 0.646]
harm_vals  = [accuracy, precision, recall, f1]

x = range(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - w/2 for i in x], toxic_vals, w, label="ToxicChat", color="steelblue")
ax.bar([i + w/2 for i in x], harm_vals,  w, label="HarmBench",  color="tomato")
ax.set_xticks(list(x)); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.0); ax.set_ylabel("Score")
ax.set_title("Guard Performance: ToxicChat vs HarmBench")
ax.legend(); ax.axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()

### Save results

In [ ]:
results = {
    "harmbench": {
        "n": n, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    },
    "toxicchat_reference": {
        "accuracy": 0.776, "precision": 0.879, "recall": 0.510, "f1": 0.646
    }
}
Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/harmbench_validation.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved artifacts/harmbench_validation.json")

### Save to Drive

In [ ]:
import shutil
from pathlib import Path

drive_dest = "/content/drive/MyDrive/hf_cache/artifacts"
Path(drive_dest).mkdir(parents=True, exist_ok=True)
shutil.copy("artifacts/harmbench_validation.json", drive_dest)
print("Saved to Drive.")

### Paper write-up (Section: Robustness to Training Data Contamination)

> *"To assess whether our reported guard metrics reflect genuine failure patterns
> rather than training-data memorisation, we evaluated Llama-Guard-3-8B on a
> 500-prompt sample from HarmBench [cite], which post-dates ToxicChat's publication
> and is unlikely to appear in the guard's training data. Precision on HarmBench was
> X% vs 87.9% on ToxicChat, and recall was Y% vs 51.0% (Table X). The consistent
> failure rates across both datasets suggest the guard's systematic blind spots
> represent genuine generalisation limitations rather than contamination artefacts,
> strengthening the motivation for prototype-driven auditing."*